
<div style="background:linear-gradient(135deg,#0B2E59 0%,#0E4C6E 55%,#00A6C0 100%);
            padding:34px 30px;border-radius:14px;color:#F4FBFF;margin-bottom:18px;">
  <div style="font-size:13px;letter-spacing:2px;color:#9FE0EA;font-weight:700;">
    TEAM3-AI &nbsp;·&nbsp; BINX TECH &nbsp;·&nbsp; CONTENT-BASED RECOMMENDATION WORKSTREAM
  </div>
  <h1 style="margin:10px 0 6px;font-size:30px;">Person-to-Person (Profile) Recommendation</h1>
  <div style="font-size:16px;color:#D7ECF5;">Phase 2 &amp; 3 — Core Build, Content Profiles, TF-IDF, Cosine Similarity &amp; Validation</div>
  <div style="margin-top:16px;">
    <span style="background:#0B2E59;padding:5px 12px;border-radius:20px;font-size:11px;font-weight:700;margin-right:6px;">OWNER · ZAYAN SHAWAREB</span>
    <span style="background:#00A6C0;padding:5px 12px;border-radius:20px;font-size:11px;font-weight:700;margin-right:6px;color:#062330;">TASK · PROFILE ↔ PROFILE</span>
    <span style="background:#FFC857;padding:5px 12px;border-radius:20px;font-size:11px;font-weight:700;margin-right:6px;color:#4A3300;">PHASE · 2-3 CORE BUILD &amp; VALIDATION</span>
    <span style="background:#1BAF7A;padding:5px 12px;border-radius:20px;font-size:11px;font-weight:700;margin-right:6px;color:#062330;">STATUS · IN PROGRESS</span>
    <span style="background:#8B5CF6;padding:5px 12px;border-radius:20px;font-size:11px;font-weight:700;color:#F4FBFF;">PLATFORM · EDUCATIONAL KNOWLEDGE SHARING</span>
  </div>
</div>



## 📖 The Story So Far

The AI research and design for this task are already finalized and agreed with the Backend team:

- **Goal:** recommend relevant people for a user to connect with, based on profile similarity across **Skills**, **Interests**, and **Learning Direction**.
- **Confirmed schema (Backend, Sept 2026):** `Skill` and `Interest` are **many-valued** per profile (via `UserSkill` / `UserInterest`), while `LearningDirection` is a **single, nullable** field per profile — a user may have zero or one current learning direction.
- **Approach:** combine each profile's fields into one **content profile** (text), vectorize with **TF-IDF**, rank candidates by **Cosine Similarity**, and return the top-N most similar profiles excluding the user themself.
- **Why synthetic data:** the platform has no real users yet (cold-start), so the model is built and validated on synthetic profiles that mirror the real database schema exactly — once real data exists, the same pipeline runs unchanged.

This notebook is where that design becomes real, working code: generating schema-matched synthetic data, building the pipeline end-to-end, and validating it against the edge cases that matter most — including the `LearningDirection = null` case the Backend team confirmed.



## 🎯 What This Notebook Covers

| # | Objective | Trello Card |
|---|---|---|
| 1 | Generate synthetic profiles matching the real DB schema | Generate Synthetic User Profiles |
| 2 | Build the combined "content profile" text, handling `null` Learning Direction | Implement Content Profile Builder |
| 3 | Vectorize content profiles with TF-IDF | Implement TF-IDF Vectorization |
| 4 | Compute pairwise Cosine Similarity | Implement Cosine Similarity |
| 5 | Rank and return top-N recommendations, excluding self | Core Build (ranking) |
| 6 | Validate: sanity check, empty/minimal profile, identical profiles, different profile counts | Phase 3 — Validation |


In [1]:
import random
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)

pd.set_option("display.max_colwidth", 120)
print("Environment ready.")


Environment ready.



## 1 · Generating Synthetic Profiles

The synthetic data mirrors the **real database schema** the Backend team defined:

| Table / Relation | Fields Used | Cardinality |
|---|---|---|
| `Profile` | `first_name`, `last_name`, `bio`, `university` | — |
| `Skill` (via `UserSkill`) | tag list | one or more per profile |
| `Interest` (via `UserInterest`) | tag list | one or more per profile |
| `LearningDirection` | single tag | **one or none** per profile (nullable) |

To make sure the pipeline is honest about real conditions from day one, roughly **15% of synthetic profiles are generated with `learning_direction = None`** — the exact null case the Backend team confirmed the API can return.


In [2]:
FIRST_NAMES = ["Layla","Omar","Sara","Yousef","Rana","Khaled","Dina","Ahmad","Nour","Tariq",
               "Hala","Zaid","Maya","Fadi","Lina","Anas","Rima","Bilal","Salma","Yazan",
               "Huda","Karim","Amal","Wassim","Reem","Nabil","Farah","Adel","Jana","Samer"]
LAST_NAMES  = ["Odeh","Hamdan","Kittaneh","Barghouti","Nasser","Salhi","Awad","Qasem","Zayed","Hourani"]
UNIVERSITIES = ["An-Najah National University","Birzeit University","Palestine Polytechnic University",
                "Arab American University","Palestine Technical University"]

SKILLS = ["Python","JavaScript","SQL","Data Analysis","UI Design","Public Speaking",
          "Project Management","Java","React","Statistics","Copywriting","Video Editing",
          "C++", "Cloud Computing", "Excel"]

INTERESTS = ["Machine Learning","Web Development","Mobile Development","Cybersecurity",
             "Game Development","UI/UX Design","Data Science","Cloud Computing",
             "Entrepreneurship","Robotics","Blockchain","Digital Marketing"]

LEARNING_DIRECTIONS = ["AI & Machine Learning","Web Development","Mobile Development",
                        "Cybersecurity","UI/UX Design","Data Science", None]
# None is included in the pool below with an explicit weight, not just uniform choice.

BIO_TEMPLATES = [
    "Passionate about {a} and always exploring {b}.",
    "{unibucket} student focused on {a}, currently diving into {b}.",
    "I enjoy building things with {a} and learning more about {b} in my free time.",
    "Curious about {a}; recently started exploring {b} through side projects.",
]

def make_profile(pid: int) -> dict:
    first = random.choice(FIRST_NAMES)
    last = random.choice(LAST_NAMES)
    uni = random.choice(UNIVERSITIES)

    n_skills = random.randint(1, 4)
    skills = random.sample(SKILLS, n_skills)

    n_interests = random.randint(1, 3)
    interests = random.sample(INTERESTS, n_interests)

    # ~15% explicit None, matching the nullable field Backend confirmed
    if random.random() < 0.15:
        learning_direction = None
    else:
        learning_direction = random.choice([ld for ld in LEARNING_DIRECTIONS if ld is not None])

    bio_topic_a = random.choice(skills + interests)
    bio_topic_b = random.choice(interests)
    bio = random.choice(BIO_TEMPLATES).format(a=bio_topic_a, b=bio_topic_b, unibucket=uni.split()[0])

    return {
        "profile_id": pid,
        "first_name": first,
        "last_name": last,
        "university": uni,
        "bio": bio,
        "skills": skills,
        "interests": interests,
        "learning_direction": learning_direction,
    }

N_PROFILES = 60
profiles = [make_profile(i) for i in range(N_PROFILES)]
profiles_df = pd.DataFrame(profiles)

n_null_ld = profiles_df["learning_direction"].isna().sum()
print(f"Generated {N_PROFILES} synthetic profiles.")
print(f"Profiles with learning_direction = None: {n_null_ld} ({n_null_ld/N_PROFILES:.0%})")
profiles_df.head(5)


Generated 60 synthetic profiles.
Profiles with learning_direction = None: 9 (15%)


,profile_id,first_name,last_name,university,bio,skills,interests,learning_direction
0,0,Huda,Hamdan,An-Najah National University,Passionate about Web Development and always exploring Web Development.,"[Data Analysis, Excel, SQL]","[Web Development, Blockchain, Entrepreneurship]",NaN
1,1,Sara,Barghouti,Birzeit University,Curious about Digital Marketing; recently started exploring Digital Marketing through side projects.,[React],[Digital Marketing],UI/UX Design
2,2,Salma,Nasser,An-Najah National University,Curious about Video Editing; recently started exploring Game Development through side projects.,"[Video Editing, Project Management]","[Game Development, Mobile Development]",Mobile Development
3,3,Yousef,Salhi,Palestine Polytechnic University,I enjoy building things with C++ and learning more about Web Development in my free time.,"[C++, Python, Video Editing]","[Entrepreneurship, Web Development]",Cybersecurity
4,4,Salma,Barghouti,An-Najah National University,I enjoy building things with Copywriting and learning more about Game Development in my free time.,[Copywriting],[Game Development],Web Development



## 2 · Building the Content Profile

Each profile's `Skills`, `Interests`, `LearningDirection`, and `bio` are combined into a single text representation — the **content profile**. Skills and interests are lists (many-valued), so they're joined into space-separated text; `learning_direction` is a single field that may be `null`, confirmed directly by the Backend team on Sept 17.

The obvious guard is `profile["learning_direction"] or ""`. It's exactly what was proposed on Discord — and it looks safe. Running it against the real synthetic data below turned up a genuine bug worth documenting rather than quietly patching.


In [3]:
def build_content_profile_naive(profile: dict) -> str:
    parts = [
        " ".join(profile["skills"]),
        " ".join(profile["interests"]),
        profile["learning_direction"] or "",      # looks safe...
        profile["bio"],
    ]
    return " ".join(p for p in parts if p).strip()

# Pull a real row that has a null learning_direction straight out of the DataFrame
null_row = profiles_df[profiles_df["learning_direction"].isna()].iloc[0].to_dict()
print(f"learning_direction value: {null_row['learning_direction']!r}   type: {type(null_row['learning_direction'])}")

try:
    build_content_profile_naive(null_row)
    print("Naive version worked (unexpected).")
except TypeError as e:
    print(f"Naive version FAILED: {e}")


learning_direction value: nan   type: <class 'float'>
Naive version FAILED: sequence item 2: expected str instance, float found



**Why it broke:** a plain Python `None` is falsy, so `None or ""` correctly becomes `""`. But once a column with mixed strings and `None` sits inside a **pandas DataFrame**, pandas silently stores the missing entries as `NaN` — a `float`, not `None`. `NaN` is a non-zero float, which Python treats as **truthy**, so `nan or ""` evaluates to `nan` itself, not `""`. `" ".join(...)` then tries to join a float and raises `TypeError`.

This is the same *training/serving skew* family of bug as Week 8 Day 4 — a value looks equivalent (`None` vs. `NaN`) but behaves differently depending on which layer of the code touches it first. The fix has to check for **both** explicitly.


In [4]:
import math

def is_missing(value) -> bool:
    # True for None, NaN (pandas' coercion of None), or an empty string.
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return value == ""

def build_content_profile(profile: dict) -> str:
    parts = [
        " ".join(profile["skills"]),
        " ".join(profile["interests"]),
        "" if is_missing(profile["learning_direction"]) else profile["learning_direction"],
        profile["bio"],
    ]
    return " ".join(p for p in parts if p).strip()

# Re-run the exact row that broke the naive version
fixed_result = build_content_profile(null_row)
print(f"Fixed version on the same null row -> {fixed_result!r}  (no error)")

profiles_df["content_profile"] = profiles_df.apply(
    lambda row: build_content_profile(row.to_dict()), axis=1
)

with_ld = profiles_df[profiles_df["learning_direction"].notna()].iloc[0]
without_ld = profiles_df[profiles_df["learning_direction"].isna()].iloc[0]

print("\nProfile WITH learning_direction:")
print(f"  learning_direction = {with_ld['learning_direction']!r}")
print(f"  content_profile    = {with_ld['content_profile']!r}\n")

print("Profile WITHOUT learning_direction (null case):")
print(f"  learning_direction = {without_ld['learning_direction']!r}")
print(f"  content_profile    = {without_ld['content_profile']!r}")


Fixed version on the same null row -> 'Data Analysis Excel SQL Web Development Blockchain Entrepreneurship Passionate about Web Development and always exploring Web Development.'  (no error)

Profile WITH learning_direction:
  learning_direction = 'UI/UX Design'
  content_profile    = 'React Digital Marketing UI/UX Design Curious about Digital Marketing; recently started exploring Digital Marketing through side projects.'

Profile WITHOUT learning_direction (null case):
  learning_direction = nan
  content_profile    = 'Data Analysis Excel SQL Web Development Blockchain Entrepreneurship Passionate about Web Development and always exploring Web Development.'


In [5]:
assert profiles_df["content_profile"].apply(lambda s: isinstance(s, str) and len(s) > 0).all()
print("All", len(profiles_df), "content profiles built successfully — including every null Learning Direction case.")


All 60 content profiles built successfully — including every null Learning Direction case.



## 3 · TF-IDF Vectorization

Each `content_profile` string is converted into a numeric vector using **TF-IDF (Term Frequency – Inverse Document Frequency)**: words that appear often in one profile but rarely across all profiles get more weight, since they're more distinctive of that person.


In [6]:
vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(profiles_df["content_profile"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}  (profiles x vocabulary terms)")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)} terms")
print("Sample terms:", list(vectorizer.vocabulary_.keys())[:10])


TF-IDF matrix shape: (60, 54)  (profiles x vocabulary terms)
Vocabulary size: 54 terms
Sample terms: ['data', 'analysis', 'excel', 'sql', 'web', 'development', 'blockchain', 'entrepreneurship', 'passionate', 'exploring']



## 4 · Cosine Similarity

Pairwise **Cosine Similarity** is computed between every pair of profile vectors — a score from 0 (nothing in common) to 1 (identical content profiles).


In [7]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print(f"Similarity matrix shape: {similarity_matrix.shape}")
diag = np.diag(similarity_matrix)
print(f"Diagonal check (a profile vs. itself) — min: {diag.min():.4f}, max: {diag.max():.4f}  (expected ~1.0)")


Similarity matrix shape: (60, 60)
Diagonal check (a profile vs. itself) — min: 1.0000, max: 1.0000  (expected ~1.0)



## 5 · Top-N Recommendation Function

Given a `profile_id`, rank every other profile by similarity score and return the top-N — **excluding the profile itself**.


In [8]:
def recommend(profile_id: int, top_n: int = 5) -> pd.DataFrame:
    idx = profiles_df.index[profiles_df["profile_id"] == profile_id][0]
    scores = similarity_matrix[idx].copy()
    scores[idx] = -1  # exclude the profile itself
    top_idx = np.argsort(scores)[::-1][:top_n]

    result = profiles_df.loc[top_idx, ["profile_id", "first_name", "last_name", "skills", "interests", "learning_direction"]].copy()
    result["similarity_score"] = scores[top_idx].round(4)
    return result.reset_index(drop=True)

sample = profiles_df.iloc[3]
print(f"Recommendations for profile #{sample['profile_id']} — {sample['first_name']} {sample['last_name']}")
print(f"  skills: {sample['skills']}, interests: {sample['interests']}, learning_direction: {sample['learning_direction']}\n")
recommend(sample["profile_id"], top_n=5)


Recommendations for profile #3 — Yousef Salhi
  skills: ['C++', 'Python', 'Video Editing'], interests: ['Entrepreneurship', 'Web Development'], learning_direction: Cybersecurity



,profile_id,first_name,last_name,skills,interests,learning_direction,similarity_score
0,51,Amal,Kittaneh,"[Video Editing, React]","[Game Development, Robotics, Entrepreneurship]",Cybersecurity,0.6069
1,36,Lina,Qasem,"[C++, Excel, Video Editing, SQL]","[Web Development, Game Development, Entrepreneurship]",UI/UX Design,0.5883
2,25,Samer,Hamdan,"[JavaScript, Excel, Video Editing]","[Mobile Development, Game Development, Blockchain]",Data Science,0.5787
3,48,Salma,Barghouti,"[C++, Public Speaking, SQL]","[Robotics, Web Development, Mobile Development]",AI & Machine Learning,0.5506
4,56,Nour,Barghouti,"[Public Speaking, JavaScript]","[Entrepreneurship, Mobile Development, Cybersecurity]",Cybersecurity,0.5455



## 6 · Validation & Edge-Case Testing

Four checks, matching the Phase 3 Trello cards:

1. **Sanity check** — do the top recommendations actually share skills/interests with the target profile?
2. **Empty / minimal profile** — a profile with almost nothing filled in must not crash the pipeline.
3. **Identical profiles** — two profiles with the exact same fields should recommend each other first, with similarity ≈ 1.0.
4. **Different profile counts** — the pipeline should run cleanly at small and larger profile counts.


### 6.1 · Sanity Check on Top-N Recommendations

In [9]:
target = profiles_df.iloc[10]
recs = recommend(target["profile_id"], top_n=3)

print(f"Target profile #{target['profile_id']}: skills={target['skills']}, interests={target['interests']}, learning_direction={target['learning_direction']}\n")

for _, r in recs.iterrows():
    shared_skills = set(target["skills"]) & set(r["skills"])
    shared_interests = set(target["interests"]) & set(r["interests"])
    same_ld = (target["learning_direction"] is not None) and (target["learning_direction"] == r["learning_direction"])
    print(f"  -> #{r['profile_id']} {r['first_name']} {r['last_name']}  score={r['similarity_score']:.4f}")
    print(f"     shared skills: {sorted(shared_skills) or 'none'} | shared interests: {sorted(shared_interests) or 'none'} | same learning_direction: {same_ld}")


Target profile #10: skills=['Java', 'Python'], interests=['Digital Marketing', 'Game Development', 'Entrepreneurship'], learning_direction=UI/UX Design

  -> #52 Hala Barghouti  score=0.5218
     shared skills: ['Java'] | shared interests: ['Digital Marketing'] | same learning_direction: False
  -> #51 Amal Kittaneh  score=0.4983
     shared skills: none | shared interests: ['Entrepreneurship', 'Game Development'] | same learning_direction: False
  -> #3 Yousef Salhi  score=0.4907
     shared skills: ['Python'] | shared interests: ['Entrepreneurship'] | same learning_direction: False


### 6.2 · Empty / Minimal Profile

In [10]:
minimal_profile = {
    "profile_id": 9001,
    "first_name": "New",
    "last_name": "User",
    "university": "",
    "bio": "",
    "skills": [],
    "interests": [],
    "learning_direction": None,
}

minimal_content = build_content_profile(minimal_profile)
print(f"Minimal profile content_profile: {minimal_content!r} (empty string is expected and valid)")

# Re-run the full pipeline with this profile appended, to prove it doesn't break anything downstream
test_df = pd.concat([profiles_df, pd.DataFrame([{**minimal_profile, "content_profile": minimal_content}])], ignore_index=True)
test_tfidf = vectorizer.fit_transform(test_df["content_profile"])
test_sim = cosine_similarity(test_tfidf)

minimal_idx = test_df.index[test_df["profile_id"] == 9001][0]
minimal_scores = test_sim[minimal_idx]
minimal_scores_excl_self = np.delete(minimal_scores, minimal_idx)

print(f"Pipeline completed without errors for the near-empty profile.")
print(f"Its similarity scores against everyone else range from {minimal_scores_excl_self.min():.4f} to {minimal_scores_excl_self.max():.4f} "
      f"(low/near-zero as expected — there is simply no text to match on).")


Minimal profile content_profile: '' (empty string is expected and valid)
Pipeline completed without errors for the near-empty profile.
Its similarity scores against everyone else range from 0.0000 to 0.0000 (low/near-zero as expected — there is simply no text to match on).


### 6.3 · Identical Profiles

In [11]:
twin_a = {
    "profile_id": 9002, "first_name": "Twin", "last_name": "A", "university": "Test University",
    "bio": "Passionate about Python and Machine Learning.",
    "skills": ["Python", "SQL"], "interests": ["Machine Learning"], "learning_direction": "AI & Machine Learning",
}
twin_b = {**twin_a, "profile_id": 9003, "first_name": "Twin", "last_name": "B"}

twin_a["content_profile"] = build_content_profile(twin_a)
twin_b["content_profile"] = build_content_profile(twin_b)

twins_df = pd.concat([profiles_df, pd.DataFrame([twin_a, twin_b])], ignore_index=True)
twins_tfidf = vectorizer.fit_transform(twins_df["content_profile"])
twins_sim = cosine_similarity(twins_tfidf)

idx_a = twins_df.index[twins_df["profile_id"] == 9002][0]
idx_b = twins_df.index[twins_df["profile_id"] == 9003][0]
mutual_score = twins_sim[idx_a, idx_b]

scores_for_a = twins_sim[idx_a].copy()
scores_for_a[idx_a] = -1
top_match_idx = np.argmax(scores_for_a)

print(f"Similarity(Twin A, Twin B) = {mutual_score:.4f}  (expected ~1.0 for identical content profiles)")
print(f"Twin A's #1 recommendation is profile #{twins_df.loc[top_match_idx, 'profile_id']} "
      f"({'Twin B — correct' if top_match_idx == idx_b else 'NOT Twin B — unexpected'})")


Similarity(Twin A, Twin B) = 1.0000  (expected ~1.0 for identical content profiles)
Twin A's #1 recommendation is profile #9003 (Twin B — correct)


### 6.4 · Different Profile Counts (Scale Check)

In [12]:
def time_pipeline(n_profiles: int) -> float:
    test_profiles = [make_profile(i) for i in range(n_profiles)]
    df = pd.DataFrame(test_profiles)
    df["content_profile"] = df.apply(lambda row: build_content_profile(row.to_dict()), axis=1)

    start = time.time()
    vec = TfidfVectorizer(lowercase=True, stop_words="english")
    matrix = vec.fit_transform(df["content_profile"])
    _ = cosine_similarity(matrix)
    elapsed = time.time() - start
    return elapsed

for n in [10, 100, 500]:
    random.seed(42)
    elapsed = time_pipeline(n)
    print(f"  N = {n:>4} profiles  ->  pipeline (vectorize + full similarity matrix) ran in {elapsed*1000:.1f} ms")

print("\nNo errors at any scale — well within range for an MVP; the current approach does not need to change before real data arrives.")


  N =   10 profiles  ->  pipeline (vectorize + full similarity matrix) ran in 1.6 ms
  N =  100 profiles  ->  pipeline (vectorize + full similarity matrix) ran in 3.7 ms
  N =  500 profiles  ->  pipeline (vectorize + full similarity matrix) ran in 10.0 ms

No errors at any scale — well within range for an MVP; the current approach does not need to change before real data arrives.



## ✅ Summary & Next Steps

**Completed in this notebook (Phases 2 & 3):**
- Schema-matched synthetic data generation, including the confirmed `learning_direction = null` case (~15% of profiles).
- Content profile builder — combines Skills, Interests, Learning Direction, and bio into one text representation. Along the way, found and fixed a real bug: pandas silently turns `None` into `NaN` (a truthy float) inside a DataFrame column, so the "obvious" `value or ""` guard passes for a raw dict but fails for a DataFrame row. The fix checks for `None`, `NaN`, and `""` explicitly.
- TF-IDF vectorization and Cosine Similarity computation.
- Top-N recommendation function, excluding the user from their own results.
- Validation: sanity-checked recommendations share real overlap in skills/interests; a near-empty profile does not break the pipeline; identical profiles correctly recommend each other (similarity ≈ 1.0); the pipeline scales cleanly from 10 to 500 profiles.

**Next (Phase 4):** lock in the `GET /recommendations` response contract with the Backend team (ranked IDs + similarity scores vs. full profile summaries), and receive the fixed taxonomy lists for Skills, Interests, and Learning Direction so the model is validated against the real categories rather than invented ones.

---

<div style="font-size:12.5px;color:#55677E;">BinX Tech · AI &amp; ML Internship — Team3-AI — Content-Based Recommendation Workstream</div>
